In [5]:
# Group suffixes into natural categories

import os
import sys
import yaml
import pandas as pd
import numpy as np

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']

sys.path.append(os.path.join(LOCAL_PATH, "src/python"))

from utils import parse_casenum

from matplotlib import pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11

os.environ['LOKY_MAX_CPU_COUNT'] = '1' # because of windows core count warning

with open('../../config.local.yaml', 'r') as f:
    local_config = yaml.safe_load(f)
with open('../../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

LOCAL_PATH = local_config['LOCAL_PATH']
DATA_PATH = local_config['DATA_PATH']
EMBEDDING_DIMENSION = config['EMBEDDING_DIMENSION']

rng = np.random.default_rng(12898)

N_CLUSTERS = 3
N_COMPONENTS = 10

In [6]:
df = pd.read_parquet(os.path.join(DATA_PATH, "intermediate_data/cpc", "analysis_data_w_distances.parquet"))

In [7]:
sfx_df = {}
for idx, row in df.iterrows():
    title = row["title"]
    parsed_casenum = parse_casenum(title)
    suffixes = parsed_casenum['suffixes']
    for sfx in suffixes:
        if sfx not in sfx_df:
            sfx_df[sfx] = 0
        sfx_df[sfx] += 1
sfx_df = pd.DataFrame(list(sfx_df.items()), columns=['suffix', 'count'])
sfx_df = sfx_df.sort_values(by='count', ascending=False)

sfx_df.to_csv(os.path.join(DATA_PATH, "intermediate_data/cpc", "suffix_counts.csv"), index=False)

In [8]:
"""
Manual step:
Take suffix_counts.csv, upload it along with "Active Prefix Suffix Codes.csv" to a LLM, and give the LLM this prompt:

---

Attached is `suffix_counts.csv` which pertains to City of Los Angeles Planning Department case number suffixes. Each suffix represents a relevant detail of the case, such as requests for zone variances, appeals, etc. 

Use your knowledge of planning and zoning in the City of Los Angeles to assign each suffix to one of the following categories:
- Site Plan Review
- Appeals
- Incentive Programs
- Conditional Use Permits
- Variances / Adjustments / Exceptions
- Compliance Review
- Code or Plan Amendments
- Development Agreements
- Other

Guidelines:
- It is acceptable if a category contains only one suffix.
- It is acceptable if a category contains multiple suffixes.
- Any project-specific request for a variance, exception, or adjustment to the zoning code or specific plan should be grouped under "Variances / Adjustments / Exceptions"
- Any project-specific request for a change in the relevant zoning category should be grouped under "Variances / Adjustments / Exceptions"
- Any non-project-specific request to amend the zoning code or planning standards should be grouped under "Code or Plan Amendments"
- Any request to waive or modify a condition of a development, or to transfer development rights should be grouped under "Development Agreements"
- Any program designed to streamline permitting or offer development bonuses should be grouped under "Incentive Programs"
- Any suffix indicating required compliance review to overlay district or specific plan standards should be grouped under "Compliance Review"
- Use the internet to help you understand the suffix if necessary

Return a csv with the same number of rows as `suffix_counts.csv` with four additional columns: `suffix_meaning`, `category`, `category_code`, and `reasoning`.
- `suffix_meaning` is a brief description of the suffix
- `category` is the name of the category, already given in the list above
- `category_code` is a 2 or 3 character alphanumeric code for the category, which you may decide
- `reasoning` is a brief explanation of your reasoning for the category assignment

---

Store the resulting csv in LOCAL_PATH/intermediate_data/cpc/suffix_groups.csv
"""
''

''